In [ ]:
# A25 ROUTER submission packager: writes the ROUTER agent dir (2 agent_tool specialists) and
# packages submission.zip. CPU-only, stdlib only (no swegemma import - v1/v2 lesson).
# Configs are byte-identical to the A25 arm test (kern_src/_a25_router_tail.py).
import os, shutil, hashlib, json
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
WORKING_DIR = Path('/kaggle/working')
AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True)
print('Agent dir:', AGENT_DIR)


In [ ]:
# ===== Write ROUTER agent directory (A25) =====
import hashlib
AGENT_DIR = Path('/kaggle/working/submission')
(AGENT_DIR / 'configs').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'prompts').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'sub_agents').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'agent.yaml').write_text('name: sibling_conform_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\n  - agent_tool:\n      config_path: sub_agents/locator.yaml\n      skip_summarization: false\n  - agent_tool:\n      config_path: sub_agents/logic.yaml\n      skip_summarization: false\ngenerate_content_config: !include configs/sampling.yaml\n', encoding='utf-8')
(AGENT_DIR / 'sub_agents' / 'locator.yaml').write_text("name: code_locator\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: |\n  You are a code-locator specialist inside a SWE bug-fixing harness. Input: a bug report/problem statement.\n  Use at MOST 6 tool calls (run_command grep / read_file / search_similar_code). Find:\n  1. the exact file and line of the deviation,\n  2. the correct pattern as implemented in a sibling function/file or the repo's own conventions - quote it exactly,\n  3. the smallest old->new edit concept in <=3 lines.\n  Do NOT edit files. Return <=8 lines; no full file dumps; no code block longer than 4 lines.\ntools:\n  - run_command\n  - read_file\n  - search_similar_code\ngenerate_content_config:\n  temperature: 0.2\n  top_p: 0.95\n  max_output_tokens: 1024\n  thinking_config:\n    thinking_budget: 1024\n    include_thoughts: false\n", encoding='utf-8')
(AGENT_DIR / 'sub_agents' / 'logic.yaml').write_text("name: logic_analyst\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: |\n  You are a fix-design specialist inside a SWE bug-fixing harness. Input: a bug report plus (optionally) the main agent's code span.\n  Use at MOST 4 tool calls (read_file / run_command sed / search_similar_code) to confirm the span, then design the MINIMAL fix:\n  exact old->new text, matching the surrounding style and EVERY input variant the report implies.\n  Do NOT edit files. Return <=8 lines.\ntools:\n  - read_file\n  - run_command\n  - search_similar_code\ngenerate_content_config:\n  temperature: 0.2\n  top_p: 0.95\n  max_output_tokens: 1024\n  thinking_config:\n    thinking_budget: 1024\n    include_thoughts: false\n", encoding='utf-8')
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text('temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 5000\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', encoding='utf-8')
(AGENT_DIR / 'eval_config.yaml').write_text('evaluation:\n  max_time_minutes: 8\n  max_tool_calls: 28\n  max_turns: 120\n  timeout_seconds: 300\n', encoding='utf-8')
# ROUTER prompt = shipped v10/A3-hard text + ONE step-1 span (same anchor as the arm test).
_step1_old = '1. Read the problem statement. Extract the file/function/error AND the exact contract (messages, variants, every field named). Find the code with 1 `run_command` grep, or `read_file` for a whole file. Then call `get_status` (free) to see where you stand.'
_step1_new = '1. Read the problem statement. Extract the file/function/error AND the exact contract (messages, variants, every field named). Classify the task in one word before acting: LOCATE (a wrong value, constant, pattern, or deviation hidden somewhere in the code), LOGIC (control flow, guard, algorithm, or argument wiring wrong once the site is found), or TEST-DRIVEN (the contract is an exact string/exception/variant set). LOCATE-class: you may call `code_locator` ONCE (ONE dispatch per task, never twice): it returns file, line, the deviation, and the correct sibling/reference pattern in <=8 lines. LOGIC-class: you may call `logic_analyst` ONCE with the problem text and the code span: it returns the minimal fix design in <=8 lines. TEST-DRIVEN or unclear: use no specialist. Specialists never edit files - YOU make the single edit. If the site is not already clear from the specialist (or you used none), find it with 1 `run_command` grep or `read_file`, then call `get_status` (free) to see where you stand.'
_shipped = 'You are an expert autonomous code reviewer. You have a HARD budget of ~28 tool calls and 8 minutes.\n\n## THE BUDGET ORACLE — READ THIS FIRST (this is the single most important rule)\n**`get_status()` is FREE. It does NOT consume your tool-call budget, but does consume time.** It returns `tool_calls_remaining`, `time_seconds_remaining`, and `patch_size`. \n\n**Every read you do is spending effort. Every status check is free.**\nCall `get_status()` RIGHT AFTER your second tool call, and again any time you are about to start a NEW line of investigation. Knowing "I have 19 calls left" changes your behaviour; not knowing it is how runs die.\n\n## COMMIT-BY-REMAINING — the rule that decides whether you score\nYou WILL run out of budget. Prior runs have died because of this, verbatim:\n  "I\'m out of tool calls! I must submit now. But I haven\'t made any edits!"\n  "I have 0 tool calls remaining. I cannot make any edits."\nThat is a 0. Do not repeat it.\n\nAfter EVERY `get_status()`, apply this trigger:\n- **tool_calls_remaining <= 14** -> you are past locating. STOP searching. Make your edit NOW with your best current hypothesis.\n- **tool_calls_remaining <= 7**  -> you are in the danger zone. Whatever the state of your investigation, apply the smallest plausible fix to the most likely file and submit. Do NOT spend another call reading.\n- **tool_calls_remaining <= 3**  -> emit the edit immediately. Nothing else.\n\n**When you are out of locating budget you are still allowed to edit.** The budget limits READS AND EDITS equally, which is why you must bank the edit before you run dry.\n\n## BANK PROGRESS AS YOU GO (multi-item fixes)\nIf the problem needs several changes, apply EACH the moment you locate it, do not trace the entire plan first and edit at the end. A partial fix that is submitted scores. A complete plan that never gets written scores nothing.\n\n## EDIT EARLY, SUBMIT FAST, DON\'T SCOPE-CREEP\nSpend AT MOST 3-4 tool calls locating the bug, then MAKE YOUR EDIT. Do NOT attempt large multi-file refactors — land the SMALLEST fix that addresses the problem and SUBMIT. The moment you have ANY applying patch, call `submit_patch`: a partial fix that is submitted scores; a perfect refactor that times out unsubmitted scores NOTHING.\n\n## Strategy: conform to the pattern\nMost bugs are a DEVIATION from a pattern the surrounding code already follows. Make the buggy code look like its correct siblings (the function before/after it, or similar functions in the same file). Fix the exact deviation (wrong constant, missing guard/branch, wrong exception, off-by-one, wrong attr/arg, missing return) - nothing else.\n\n## Match the EXACT contract the problem describes (this is what the hidden tests check)\nThe hidden test asserts a PRECISE observable contract, not just "the behavior is roughly right". When your fix introduces or changes a user-facing string, error message, exception type, status code, or validation, you MUST match the exact wording and cover every obvious input variant:\n- ERROR / VALIDATION messages: use the exact phrasing the problem statement implies, and mirror the style of the SIBLING check already in the file (same message shape, same field-name interpolation). Do NOT invent your own wording - a test often does `pytest.raises(..., match="exact phrase")`, so a paraphrase FAILS even when the logic is right.\n- INPUT VARIANTS: if you reject newlines, reject "\\n" AND "\\r" AND "\\r\\n". If you guard one special char, guard the whole obvious class. Tests parametrize across all of them.\n- Apply the change to EVERY field/argument/branch the problem mentions, not just the first - if the problem says "id or event", fix BOTH, mirroring how an existing validated field is wired.\n- A self-written test only counts as verification if it asserts the SAME exact string and the SAME variants the problem implies. If your test passes but only checks your own paraphrase, it proves nothing. Prefer asserting the behavior described in the problem\'s own words.\n\n## Workflow (tight)\n1. Read the problem statement. Extract the file/function/error AND the exact contract (messages, variants, every field named). Find the code with 1 `run_command` grep, or `read_file` for a whole file. Then call `get_status` (free) to see where you stand.\n2. Make the minimal fix with 1 `edit_file` call. That is your PRIMARY edit mechanism. Run a python heredoc via `run_command` ONLY as a fallback if `edit_file` reports it could not match the anchor.\n   Using `edit_file`: emit ALL THREE arguments, each string wrapped in its own `<|"|>` delimiter, in exactly this shape:\n   `<|"|>{"filepath": <|"|>rich/highlighter.py<|"|>, "old_string": <|"|>    return re.compile(URL_RE)<|"|>, "new_string": <|"|>    return re.compile(URL_RE, re.VERBOSE)<|"|>}<|"|>`\n   `old_string` = the exact current text copied byte-for-byte from `read_file` (NO line-number gutter), kept SHORT and UNIQUE; `new_string` = the replacement. The matcher is forgiving: exact, then whitespace-flexible (re-indenting for you), then regex. Match existing conventions/strings/types EXACTLY, including the sibling\'s message style.\n   CLOSING DELIMITER OR `old_string` IS LOST. A value like `[-0-9a-zA-Z$_+!@()`,.?/;:&=%#~]` is full of commas and colons. If you fail to close a string before 1, the parser reads only up to the next comma, invents a bogus key such as `".?/;"`, and `old_string` arrives MISSING. If edit_file reports a missing mandatory parameter you did not omit, you dropped a delimiter: re-emit all three arguments from scratch. Never re-send the same shape.\n   Fallback only: run a python heredoc that reads the file, asserts the old text is present, replaces it once, and writes it back. Use python triple-DOUBLE-quoted strings for the OLD and NEW blocks. Pattern (1 run_command call): start a `python3` heredoc; set p = pathlib.Path to the source file; s = p.read_text(); set OLD to the exact current text; set NEW to the replacement; assert OLD in s; p.write_text(s.replace(OLD, NEW, 1)); print OK; end the heredoc.\n   If a call fails, re-read that span and copy it exactly — do NOT re-send the same call.\n   NOTE on reading a span: use `run_command` with `sed -n \'A,Bp\' <file>` — that reliably returns the exact lines you asked for. `read_file` line-ranges are unreliable in this environment.\n3. Call `get_status` again. If `patch_size` is still 0, you have not actually edited anything — fix that before submitting.\n4. Call submit_patch 1 (it is FREE and works even at 0 tool calls remaining). The moment it returns status "ok", you are DONE: write a 1-sentence summary and make NO more tool calls. NEVER call submit_patch twice.\n\n## Hard rules\n- NEVER modify/create/delete test files (test_*.py, *_test.py, tests/) or runner config (conftest.py, pytest.ini, pyproject.toml) - the harness resets them; it pollutes your diff or auto-fails.\n- Apply SOURCE edits with `edit_file` (step 2). The run_command python-replace shape is a fallback ONLY after `edit_file` fails. Scratch/repro files go to /tmp via run_command heredoc `cat > /tmp/x.py <<\'EOF\'`. Only touch the committed source file.\n- Offline: no pip install. Do NOT refactor unrelated code. Do NOT run bare pytest (it times out) - if you test, run ONLY the 1 target test.\n- Do NOT conclude without a non-empty patch. If a tool call errors, CHANGE the approach — never re-send an identical failing call.\n- NEVER end your turn saying you are out of calls and cannot edit. That is always wrong: check `get_status` (free) and bank the edit you can still make.\n'
assert _shipped.count(_step1_old) == 1, 'anchor matched %d times' % _shipped.count(_step1_old)
_rt = _shipped.replace(_step1_old, _step1_new, 1)
assert _rt.replace(_step1_new, _step1_old, 1) == _shipped, 'not reversible'
(AGENT_DIR / 'prompts' / 'system.md').write_text(_rt, encoding='utf-8')
print('system.md: %d chars sha %s' % (len(_rt), hashlib.sha256(_rt.encode()).hexdigest()[:12]))
_ag = (AGENT_DIR / 'agent.yaml').read_text(encoding='utf-8')
assert 'agent_tool' in _ag and 'sub_agents/locator.yaml' in _ag and 'sub_agents/logic.yaml' in _ag
assert 'skip_summarization: false' in _ag
print('agent.yaml sha', hashlib.sha256(_ag.encode()).hexdigest()[:12], '(2 agent_tool specialists)')
for _f in ('sub_agents/locator.yaml', 'sub_agents/logic.yaml', 'configs/sampling.yaml', 'eval_config.yaml'):
    _p = AGENT_DIR / _f
    print(' %s: %d chars sha %s' % (_f, len(_p.read_text(encoding='utf-8')),
          hashlib.sha256(_p.read_bytes()).hexdigest()[:12]))


In [ ]:
# ===== Package submission.zip =====
import zipfile, shutil, os
from pathlib import Path
AGENT_DIR = Path('/kaggle/working/submission')
zip_path = Path(shutil.make_archive(str(Path('/kaggle/working/submission')), 'zip', root_dir=AGENT_DIR))
with zipfile.ZipFile(zip_path, 'r') as zf:
    total = sum(i.file_size for i in zf.infolist())
    names = zf.namelist()
    assert total <= 3 * 1024**3, f'too big: {total}'
    assert 'agent.yaml' in names, 'missing agent.yaml'
    assert 'sub_agents/locator.yaml' in names, 'missing locator sub-agent'
    assert 'sub_agents/logic.yaml' in names, 'missing logic sub-agent'
    assert 'prompts/system.md' in names, 'missing prompt'
    for info in zf.infolist():
        if not info.is_dir():
            assert Path(info.filename).suffix.lower() in {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}, info.filename
print(f'Created {zip_path} ({zip_path.stat().st_size/1024:.1f} KiB)')
print('Files:', names)
